# Start YOLO Pose Export / Fine-Tune Task

This notebook triggers the CHIMP training API task for the `YOLO Pose` plugin and polls task status until it finishes.


In [5]:
import json
import os
import time
import requests

In [6]:
# --- Configure run options ---
TRAINING_SERVER_URL = os.environ.get("TRAINING_SERVER_URL", "http://localhost:5253")
PLUGIN_ROUTE_NAME = "YOLO+Pose"
RUN_URL = f"{TRAINING_SERVER_URL}/tasks/run/{PLUGIN_ROUTE_NAME}"

# Required by plugin
EXPERIMENT_NAME = "yolo_pose_demo"

DATASET_NAME = "hpe_one_image"

# Polling settings
POLL_INTERVAL_SECONDS = 2
POLL_TIMEOUT_SECONDS = 900

print("Training server:", TRAINING_SERVER_URL)
print("Run URL:", RUN_URL)
print("Experiment:", EXPERIMENT_NAME)
print("Dataset:", DATASET_NAME)

Training server: http://localhost:5253
Run URL: http://localhost:5253/tasks/run/YOLO+Pose
Experiment: yolo_pose_demo
Dataset: hpe_one_image


In [7]:
# Start task
form_data = {"experiment_name": EXPERIMENT_NAME}
if DATASET_NAME:
    form_data["dataset_name"] = DATASET_NAME

run_response = requests.post(RUN_URL, data=form_data, timeout=60)
print("Start status code:", run_response.status_code)

try:
    run_payload = run_response.json()
except Exception:
    run_payload = {"raw_text": run_response.text}

print(json.dumps(run_payload, indent=2))

if run_response.status_code != 200:
    raise RuntimeError("Failed to start YOLO task. See payload above.")

task_id = run_payload.get("task_id")
if not task_id:
    raise RuntimeError("No task_id returned by training API.")

POLL_URL = f"{TRAINING_SERVER_URL}/tasks/poll/{task_id}"
print("Task ID:", task_id)
print("Poll URL:", POLL_URL)

Start status code: 200
{
  "status": "task started successfully, use '/tasks/poll/9529aa84-9423-4c2b-9ed8-8d5c99224ee6' to poll for the current status",
  "task_id": "9529aa84-9423-4c2b-9ed8-8d5c99224ee6"
}
Task ID: 9529aa84-9423-4c2b-9ed8-8d5c99224ee6
Poll URL: http://localhost:5253/tasks/poll/9529aa84-9423-4c2b-9ed8-8d5c99224ee6


In [ ]:
# Poll until task reaches a terminal state
terminal_states = {"success", "successful", "failed", "failure", "error", "done", "completed"}
start_ts = time.time()
last_payload = None

while True:
    poll_response = requests.get(POLL_URL, timeout=60)
    try:
        poll_payload = poll_response.json()
    except Exception:
        poll_payload = {"raw_text": poll_response.text}

    last_payload = poll_payload
    print(json.dumps(poll_payload, indent=2))

    if bool(poll_payload.get("ready", False)):
        break


    if time.time() - start_ts > POLL_TIMEOUT_SECONDS:
        raise TimeoutError("Polling timed out before reaching terminal state.")

    time.sleep(POLL_INTERVAL_SECONDS)

{
  "ready": false,
  "successful": null,
  "value": null
}


{
  "ready": false,
  "successful": null,
  "value": null
}
{
  "ready": false,
  "successful": null,
  "value": null
}
{
  "ready": false,
  "successful": null,
  "value": null
}
{
  "ready": false,
  "successful": null,
  "value": null
}
{
  "ready": false,
  "successful": null,
  "value": null
}
{
  "ready": false,
  "successful": null,
  "value": null
}
{
  "ready": false,
  "successful": null,
  "value": null
}
{
  "ready": false,
  "successful": null,
  "value": null
}
{
  "ready": false,
  "successful": null,
  "value": null
}
{
  "ready": true,
  "successful": true,
  "value": "20260401130146_YOLO Pose_d9e3707096594af48b62514b3136394c"
}
